---
bibliography: imbalanced_data_ref.bib
csl: apa.csl
---


![](https://raw.githubusercontent.com/EcoCommons-Australia-2024-2026/ec-notebook_site/main/images/notebooks_banner_withframe.png)

# Part 1: Imbalanced species data, background points, and weighting methods

Author details: Xiang Zhao

Editor details: Dr Sebastian Lopez Marcano

Contact details: support\@ecocommons.org.au

Copyright statement: This script is the product of the EcoCommons platform. Please refer to the EcoCommons website for more details: <https://www.ecocommons.org.au/>

Date: February 2025

# Script and data info

This notebook, developed by the EcoCommons team, showcases how to deal with imbalanced species datasets (presence and absence data) with weighting methods.

# Abstract (Part 1 and Part 2)

When building species distribution models, it is common to encounter imbalanced datasets. This may involve a large number of presence records and relatively few absence records, or, in the case of rare species, a substantial number of absence records but only a limited number of presence records.

When working with imbalanced datasets, models and predictions can be prone to several issues, including bias towards the majority class, misleading performance metrics, and reduced sensitivity to the minority class. This occurs because the model learns predominantly from the more frequent class, leading to poor generalisation for rare events [@menardi2014training]. Therefore, it is essential to address dataset imbalance to ensure fair and reliable predictions.

This notebook explores strategies to address the challenges posed by imbalanced species data in ecological modeling, particularly in Species Distribution Models (SDMs). It focuses on the use of background points and weighting methods to improve model performance when presence and absence data are disproportionate. This notebook examines the effectiveness of different pseudo-absence generation techniques, including random and buffered-random background point sampling, and applies weighting schemes to mitigate class imbalance. Using the Gang-gang Cockatoo (*Callocephalon fimbriatum*) as a case study, the analysis demonstrates the impact of these methods on model evaluation metrics such as AUROC, AUPR, F1-score, and TSS. This guide aims to equip ecologists and data scientists with practical approaches for handling imbalanced datasets, ensuring more accurate and reliable SDMs within the EcoCommons platform.

# Objectives (Part 1)

1.  Understand what imbalanced datasets are and their influence on model performance
2.  Understand how weighting methods can help address the issue of imbalanced datasets.

# **Workflow Overview**:

-   Set the working directory and load the necessary R packages. Create directories to store raw data files.

-   Data Download: Download prepared imbalanced dataset(s) from the EcoCommons Google Drive.

-   Data preparation and split.

-   Model evaluation and comparison.

In the near future, this material may form part of comprehensive support materials available to EcoCommons users. If you have any corrections or suggestions to improve the effeciengy, please [contact the EcoCommons](mailto:support@ecocommons.org.au) team.

![](https://raw.githubusercontent.com/EcoCommons-Australia-2024-2026/ec-notebook_site/main/images/EC_section_break.png)

# Set-up: R Environment and Packages

Some housekeeping before we start. This process might take some time as many packages needed to be installed.

## S.1 Set the working directory and create a folder for data.

Save the Quarto Markdown file (.QMD) to a folder of your choice, and then set the path to your folder as your working directory.


In [ ]:
# Set the workspace to the current working directory

workspace <- getwd()  # Get the current working directory and store it in 'workspace'

# Increase the plot size by adjusting the options for plot dimensions in the notebook output
options(repr.plot.width = 16, repr.plot.height = 8)  # Sets width to 16 and height to 8 for larger plots

Ideally, you would use the **`renv`** package to create an isolated environment for installing all the required R packages used in this notebook. However, since installing **`renv`** and its dependencies can be time-consuming, we recommend trying this after the workshop.

In [ ]:
# # Ensure "renv" package is installed
# if (!requireNamespace("renv", quietly = TRUE)) {
#   install.packages("renv")
# }
# 
# # Check if renv has been initialised in the project
# if (!file.exists("renv/activate.R")) {
#   message("renv has not been initiated in this project. Initializing now...")
#   renv::init()  # Initialise renv if not already set up
# } else {
#   source("renv/activate.R")  # Activate the renv environment
#   message("renv is activated.")
# }
# 
# # Check for the existence of renv.lock and restore the environment
# if (file.exists("renv.lock")) {
#   message("Restoring renv environment from renv.lock...")
#   renv::restore()
# } else {
#   message("No renv.lock file found in the current directory. Skipping restore.")
# }


## S.2 Install and load essential libraries.

Install and load R packages.

In [ ]:
# Set CRAN mirror
options(repos = c(CRAN = "https://cran.rstudio.com/"))

# List of packages to check, install if needed, and load
packages <- c("dplyr", "reshape2", "terra", "sf", "googledrive", "ggplot2", "corrplot", 
              "pROC", "dismo", "spatstat.geom", "patchwork", "biomod2", "PRROC",
              "leaflet", "car", "gridExtra", "htmltools", "RColorBrewer", "kableExtra", "tibble")

# Function to display a cat message
cat_message <- function(pkg, message_type) {
  if (message_type == "installed") {
    cat(paste0(pkg, " has been installed successfully!\n"))
  } else if (message_type == "loading") {
    cat(paste0(pkg, " is already installed and has been loaded!\n"))
  }
}

# Install missing packages and load them
for (pkg in packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
    cat_message(pkg, "installed")
  } else {
    cat_message(pkg, "loading")
  }
  library(pkg, character.only = TRUE)
}


# If you are using renv, you can snapshot the renv after loading all the packages.

#renv::snapshot()

## S.3 Download case study datasets

We have prepared the following data and uploaded them to our Google Drive for your use:

-   **Species occurrence data:** Shapefile format (.shp)

-   **Environmental variables:** Stacked Raster format (.tif)

-   **Study area boundary:** Shapefile format (.shp)

In [ ]:
# De-authenticate Google Drive to access public files
drive_deauth()

# Define Google Drive file ID and the path for downloading
zip_file_id <- "1GXOA330Ow2F7NqxdhuBfbz0a-CMT8DTb" 

datafolder_path <- file.path(workspace)

# Create a local path for the zipped file
zip_file_path <- file.path(datafolder_path, "imbalanced_data.zip")

# Function to download a file with progress messages
download_zip_file <- function(file_id, file_path) {
  cat("Downloading zipped file...\n")
  drive_download(as_id(file_id), path = file_path, overwrite = TRUE)
  cat("Downloaded zipped file to:", file_path, "\n")
}

# Create local directory if it doesn't exist
if (!dir.exists(datafolder_path)) {
  dir.create(datafolder_path, recursive = TRUE)
}

# Download the zipped file
cat("Starting to download the zipped file...\n")
download_zip_file(zip_file_id, zip_file_path)

# Unzip the downloaded file
cat("Unzipping the file...\n")
unzip(zip_file_path, exdir = datafolder_path)
cat("Unzipped files to folder:", datafolder_path, "\n")

# 1. Introduction

## 1.1 Imbalanced binary species dataset

Imbalanced datasets are common. As shown in the figure below, a dataset is considered imbalanced when the classifications are not equally represented [@chawla2002smote].

![](https://raw.githubusercontent.com/EcoCommons-Australia-2024-2026/ec-notebook_site/main/images/imbalanceddataillustration.png)

In binary species records, where data consists of presence and absence observations, a common challenge is dataset imbalance—specifically, an overabundance of presence records. This phenomenon is prevalent in many biodiversity data portals, where most species have only presence records and lack corresponding absence data.

When using an imbalanced dataset, modelling and prediction can lead to biased outcomes. For example, if presence records dominate the training set, the model may be more likely to predict presence to minimise overall errors. This can result in high sensitivity (true positive rate, correctly identifying most presences) but low specificity (true negative rate, failing to identify absences) [@johnson2012species].  

Therefore, addressing dataset imbalance before fitting SDMs is a crucial step to ensure reliable predictions.

## 1.2 Weighting methods

Many studies have shown that weighting methods can improve SDM performance [@zhang2020does; @benkendorf2023correcting; @chen2004using]. By applying class weights to SDM algorithms, the cost of misclassifying the minority class can be increased relative to the cost of misclassifying the majority class [@benkendorf2023correcting].  

For example, if a dataset contains 900 presences and 100 absences, the class ratio is 9:1. To compensate for this imbalance, class weights can be assigned inversely proportional to class frequencies. In this case, a weight of 1 would be assigned to the majority class (presences) and a weight of 9 to the minority class (absences). This means the model incurs a 9× cost for misclassifying an absence compared to a presence. Such reweighting has been shown to improve the tradeoff between model sensitivity and specificity. Specifically, while increases in sensitivity often come at the expense of specificity and overall accuracy, applying class weights can help achieve a better balance.  

In this notebook, we use a simple weighting method employed by Elith et al. [@elith2006novel] and Valavi et al. [@valavi2022predictive]. The weights are assigned by giving a weight of 1 to each presence location and adjusting the background weights so that the total weight of presence and background samples is equal.

In [ ]:
# Load necessary library
library(dplyr)

# 1️⃣ Create a toy dataset
set.seed(123)  # For reproducibility

# Example data frame with presence (1) and background/absence (0)
train_data <- data.frame(
  id = 1:10,
  occrrnS = c(rep(1, 7), rep(0, 3))  # 7 presence points, 3 background points
)

# 2️⃣ Calculate weights
# Count the number of presence and background points
prNum <- sum(train_data$occrrnS == 1)
bgNum <- sum(train_data$occrrnS == 0)

# Apply weights: presence = 1, background = prNum / bgNum
wt <- ifelse(train_data$occrrnS == 1, 1, prNum / bgNum)

# Add weights to the data frame
train_data <- train_data %>%
  mutate(weight = wt)

# 3️⃣ Display the final dataset with weights
print(train_data)

The sum of the weights for 1 (Present) is 7 × 1 = 7, which is equal to the sum of the weights for 0 (Absent), 2.33 × 3 = 7. This weighting method will be used to address the imbalanced dataset issue in this notebook.

## 1.4 Evaluation metrics

Several studies suggest how to evaluate the model performance of imbalanced dataset [@johnson2012species; @zhang2020does; @gaul2022modelling; @barbet2012selecting]. We select these to use in this notebook:

| Metric | What it Tells You | Optimised When |
|----|----|----|
| **AUROC** (Area Under ROC Curve) | Ability of the model to distinguish between classes | When overall discriminative power is important |
| **CORR** (Correlation between the observation in the occurrence dataset and the prediction) | Degree of correlation between predicted and observed values | When assessing the strength and direction of relationships |
| **Precision** | Proportion of positive predictions that are correct | False positives need to be minimised |
| **Recall** | Ability to capture actual positives | Missing positives has severe consequences |
| **Specificity** | Ability to correctly identify negatives | False positives need to be minimised |
| **F1-Score** | Balance between Precision and Recall | Both false positives and false negatives are costly |
| **AUPR** (Area Under Precision-Recall Curve) | Performance on imbalanced datasets focusing on positive class | When positive class is rare and critical |
| **TSS** (True Skill Statistic) | Skill of the model beyond random chance | When evaluating presence/absence models in ecological studies |

: We also provide some information on model evaluation on our platform's [educational material page](https://support.ecocommons.org.au/support/solutions/articles/6000163162-model-evaluation).

# 2 Overview and Conceptualisation

## 2.1 Model and notebook objective

⚠️This notebook focuses not on creating a perfect Species Distribution Model (SDM) for the Gang-gang Cockatoo, but on comparing different methods for addressing class imbalance in binary species datasets and evaluating their effectiveness in improving SDM performance.⚠️

## 2.2 Taxon, location, data, scale, and model algorithm

**Taxon:** Gang-gang Cockatoo (*Callocephalon fimbriatum*)

![](https://github.com/EcoCommons-Australia-2024-2026/ec-notebook_site_materials/raw/main/images/gang-gang_ala.jpeg){alt="Gang-gang"}

Photographer: [Kym Nicolson, ALA](https://images.ala.org.au/image/details?imageId=2433eacc-ab65-483b-af69-1207eeb41463)

The **Gang-gang Cockatoo (*Callocephalon fimbriatum*)** is endemic to southeastern Australia, with its distribution primarily spanning higher elevations and southern latitudes, including regions in New South Wales, Victoria, and the Australian Capital Territory. It inhabits temperate eucalypt forests and woodlands, favoring wet sclerophyll forests with dense acacia and banksia understories during the summer months, while moving to drier, open woodlands at lower altitudes in winter. The species is well-adapted to cooler climates and often forages in parks and suburban gardens.

Gang-gang Cockatoos face significant threats, including habitat loss due to land clearing, wildfire damage, and competition for nesting hollows with other species. Climate change, particularly increased fire frequency and altered rainfall patterns, has introduced additional risks to their survival. The species has experienced a drastic population decline, further exacerbated by the 2019–2020 bushfires, which burned approximately 28–36% of its habitat, leading to a projected long-term decline in population size.

**Location:** Australian Capital Territory (ACT, study area)

**Spatial and temporal scales:** small (spatial) and static (temporal)

**Model algorithm:** Generalised linear model (GLM)

## 2.3 Imbalanced biodiversity data

Understanding your species is essential. This includes knowing their common names (which may include multiple names) and scientific name to ensure you collect the most comprehensive records available in open-access biodiversity data portals, such as the Atlas of Living Australia (ALA) or the Global Biodiversity Information Facility (GBIF).

For this exercise, we have prepared a species occurrence data file in CSV format, which was downloaded from ALA. To make it accessible, we have stored this file in the EcoCommons Google Drive for you to download and use conveniently.

In [ ]:
# Read the shapefile for Gang-gang Cockatoo occurrence point dataset
gang_gang_act <- st_read("imbalanced_data/gang_gang_ACT.shp")

# Read the shapefile for ACT dataset
ACT <- st_read("imbalanced_data/ACT_4283.shp")

# Create leaflet map
leaflet() %>%
  addProviderTiles(providers$Esri.WorldImagery) %>%
  
  # Add the ACT layer with a distinct color
  addPolygons(
    data = ACT,
    color = "lightblue",         # Border color of ACT polygon
    weight = 1,                  # Border width
    fillColor = "lightblue",      # Fill color of ACT
    fillOpacity = 0.3,           # Transparency for fill
    group = "ACT"
  ) %>%
  
  # Add Gang-gang Cockatoo presence/absence points from the same shapefile
  addCircleMarkers(
    data = gang_gang_act,
    color = ~ifelse(occrrnS == "PRESENT", "#11aa96", "#f6aa70"),  # Dynamically set color based on occrrnS
    radius = 1,
    weight = 0.5,
    opacity = 1,
    fillOpacity = 1,
    group = "Gang-gang Cockatoo Records"
  ) %>%
  
  setView(lng = 149, lat = -35.5, zoom = 8) %>% # Set the view to desired location
  
  # Add layer controls for toggling
  addLayersControl(
    overlayGroups = c("ACT", "Gang-gang Cockatoo Records"),
    options = layersControlOptions(collapsed = FALSE)
  ) %>%
  
  # Add a legend for the layers
  addControl(
    html = "
    <div style='background-color: white; padding: 10px; border-radius: 5px;'>
      <strong>Legend</strong><br>
      <i style='background: lightblue; width: 18px; height: 18px; display: inline-block; margin-right: 8px; opacity: 0.7;'></i>
      ACT Boundary<br>
      <i style='background: #11aa96; width: 10px; height: 10px; border-radius: 50%; display: inline-block; margin-right: 8px;'></i>
      Gang-gang Cockatoo Presence<br>
      <i style='background: #f6aa70; width: 10px; height: 10px; border-radius: 50%; display: inline-block; margin-right: 8px;'></i>
      Gang-gang Cockatoo Absence
    </div>
    ",
    position = "bottomright"
  )

Now, let's calculate the number of presences and absences in the dataset to assess the extent of imbalance.

In [ ]:
# Calculate counts of presence (PRESENT) and absence (ABSENT)
occ_counts <- table(gang_gang_act$occrrnS)

# Convert to data frame for plotting
occ_df <- as.data.frame(occ_counts)
colnames(occ_df) <- c("Occurrence", "Count")

# Create bar plot with custom colors
ggplot(occ_df, aes(x = Occurrence, y = Count, fill = Occurrence)) +
  geom_bar(stat = "identity") +
  geom_text(
    aes(label = Count),
    vjust = -0.3,  
    size = 3
  ) +
  labs(
    title = "Presence vs Absence Count",
    x = "Occurrence",
    y = "Count"
  ) +
  scale_fill_manual(values = c("PRESENT" = "#11aa96", "ABSENT" = "#f6aa70")) +
  theme_minimal()
  

We can see the amount of Gang-gang Cockatoo absent records are way less than its present records.

## 2.4 Environmental variables

We conducted several tests and selected seven environmental variables for this case study. For more information on preparing environmental datasets and selecting environmental variables, please refer to our notebooks: [Raster-preparation](https://github.com/ecocommonsaustralia/notebooks/blob/main/notebooks/raster_preparation.qmd) and [GLM](https://github.com/ecocommonsaustralia/notebooks/blob/main/notebooks/EC_GLM.qmd) (section 2.3-3.1).

| Variable code | **Description** |
|----|----|
| AusClim_bioclim_03_9s_1976-2005 | **Isothermality** (BIO3) — Mean Diurnal Range / Annual Temp Range × 100 |
| AusClim_bioclim_06_9s_1976-2005 | **Min Temperature of Coldest Month** (BIO6) |
| AusClim_bioclim_09_9s_1976-2005 | **Mean Temperature of Driest Quarter** (BIO9) |
| AusClim_bioclim_15_9s_1976-2005 | **Precipitation Seasonality** (Coefficient of Variation) (BIO15) |
| AusClim_bioclim_24_9s_1976-2005 | **Annual Mean Moisture Index** (custom or derived bioclim variable) |
| AusClim_bioclim_27_9s_1976-2005 | **Mean Moisture Index of Driest Quarter** (custom/derived) |
| AusClim_bioclim_29_9s_1976-2005 | **Mean Radiation of Warmest Quarter** (custom/derived) |

In [ ]:
# Load the stacked raster layers
env_var_stack <- rast("imbalanced_data/ACT_raster.tif")

# Select the desired layers by their names
selected_layers <- c("AusClim_bioclim_03_9s_1976-2005",
                     "AusClim_bioclim_06_9s_1976-2005",
                     "AusClim_bioclim_09_9s_1976-2005",
                     "AusClim_bioclim_15_9s_1976-2005",
                     "AusClim_bioclim_24_9s_1976-2005",
                     "AusClim_bioclim_27_9s_1976-2005",
                     "AusClim_bioclim_29_9s_1976-2005")

# Subset the raster stack
selected_rasters <- env_var_stack[[selected_layers]]

# Plot all selected layers
plot(selected_rasters, main = "Selected Environmental Variables", cex.main = 0.8)
```

# 3. When we have true-absence data

## 3.1 Including true absence in the training dataset

Including true-absence data in training dataset is always recommended when such data is available [@vaclavik2009invasive]. In this study, we aim to explore the question: '**how much true-absence data is enough?**' **'Is more always better?'**

To answer this question, we allocate 90%, 70%, 50% of the true absence data, along with an equal amount of present data to the test dataset, using the rest data as training dataset.

Here, we can make a function to split the data as we want.